# CIFAR-10 PyTorch MLP 알고리즘 최적화 버전

이 파일은 원본 제출 파일을 수정하지 않고 만든 새 제출용 노트북입니다.

이번 버전의 고정 조건은 다음과 같습니다.

- MLP 모델 크기 유지: `3072 -> 2500 -> 1500 -> 500 -> 10`
- 배치 크기 유지: `batch_size = 50`
- 학습 epoch 변경: `epochs = 50`

속도 개선은 모델 크기나 배치 크기를 바꾸지 않고, 학습 알고리즘/루프 쪽에서만 적용했습니다.

- `transforms.ToTensor()` 반복 호출 제거: CIFAR-10을 처음 한 번만 텐서로 변환해 캐싱
- DataLoader 대신 텐서 미니배치 루프 사용: 작은 배치에서 발생하는 Python/DataLoader 오버헤드 감소
- CUDA 사용 시 데이터셋 GPU 캐싱 시도: 가능하면 매 배치 CPU -> GPU 복사 제거
- Adam 대신 SGD + Momentum + Nesterov + CosineAnnealingLR 사용: 1,200만 개 파라미터에서 optimizer step 비용 감소
- CUDA 사용 시 AMP 사용: mixed precision으로 행렬곱 속도 개선
- `zero_grad(set_to_none=True)`, `torch.inference_mode()` 사용
- 가능 환경에서만 `torch.compile` 자동 시도: 실패하면 원본 eager 모드로 안전하게 진행

주의: epoch를 50으로 늘리면 총 연산량은 원본 10 epoch보다 많습니다. 따라서 “동일 모델/동일 배치/50 epoch” 조건에서는 극적인 단축은 어렵지만, 불필요한 데이터 변환과 optimizer/전송 오버헤드를 줄여 같은 조건에서 더 빠르게 돌도록 구성했습니다.


## 1. 라이브러리와 실행 환경 설정


In [ ]:
import copy
import os
import time
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets

import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# CPU 스레드를 너무 많이 쓰면 작은 배치 MLP에서는 오히려 느려질 수 있습니다.
torch.set_num_threads(min(8, os.cpu_count() or 1))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    if hasattr(torch, 'set_float32_matmul_precision'):
        torch.set_float32_matmul_precision('high')

print('사용 장치:', device)
print('CPU threads:', torch.get_num_threads())
print('AMP 사용:', use_amp)


## 2. 원본 유지 설정

모델 크기와 배치 크기는 원본과 동일하게 유지하고, epoch만 50으로 설정합니다.


In [ ]:
DATA_ROOT = './data'
TRAIN_SIZE = 40_000
batch_size = 50      # 원본 유지
epochs = 50          # 요청 조건
EVAL_EVERY = 5       # 검증은 5 epoch마다 수행해 평가 오버헤드를 줄입니다.


## 3. CIFAR-10 텐서 캐싱

원본은 DataLoader가 샘플을 꺼낼 때마다 `ToTensor()` 변환을 수행합니다. CIFAR-10은 작으므로 처음 한 번만 float tensor로 바꿔두면 50 epoch 동안 반복되는 변환 비용을 줄일 수 있습니다.


In [ ]:
def cifar10_to_flat_tensors(dataset):
    images = torch.from_numpy(dataset.data).permute(0, 3, 1, 2).contiguous()
    images = images.float().div_(255.0)
    images = images.view(images.size(0), -1).contiguous()
    labels = torch.as_tensor(dataset.targets, dtype=torch.long)
    return images, labels

train_raw = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True)
test_raw = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True)
class_names = train_raw.classes

full_images, full_labels = cifar10_to_flat_tensors(train_raw)
test_images, test_labels = cifar10_to_flat_tensors(test_raw)

split_generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(full_labels), generator=split_generator)
train_indices = indices[:TRAIN_SIZE]
val_indices = indices[TRAIN_SIZE:]

train_images = full_images.index_select(0, train_indices).contiguous()
train_labels = full_labels.index_select(0, train_indices).contiguous()
val_images = full_images.index_select(0, val_indices).contiguous()
val_labels = full_labels.index_select(0, val_indices).contiguous()

# 원본 full tensor는 더 이상 필요하지 않으므로 참조를 제거합니다.
del full_images, full_labels

print('훈련 데이터 개수:', len(train_labels))
print('검증 데이터 개수:', len(val_labels))
print('테스트 데이터 개수:', len(test_labels))
print('클래스 이름:', class_names)
print('이미지 tensor shape:', train_images.shape)


## 4. 데이터셋 GPU 캐싱 시도

CUDA 메모리가 충분하면 훈련/검증/테스트 텐서를 GPU에 올려 매 배치마다 데이터를 복사하는 비용을 줄입니다. 메모리가 부족하면 자동으로 CPU 텐서 상태로 진행합니다.


In [ ]:
def tensor_bytes(*tensors):
    return sum(t.numel() * t.element_size() for t in tensors)

cache_data_on_device = False
if device.type == 'cuda':
    required_bytes = tensor_bytes(
        train_images, train_labels,
        val_images, val_labels,
        test_images, test_labels,
    )
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f'CUDA free memory: {free_bytes / 1024**3:.2f} GB')
    print(f'Dataset tensor memory: {required_bytes / 1024**3:.2f} GB')

    if free_bytes > required_bytes * 1.8:
        train_images = train_images.to(device)
        train_labels = train_labels.to(device)
        val_images = val_images.to(device)
        val_labels = val_labels.to(device)
        test_images = test_images.to(device)
        test_labels = test_labels.to(device)
        cache_data_on_device = True
        print('데이터셋을 GPU에 캐싱했습니다.')
    else:
        print('GPU 메모리 여유가 부족해 데이터셋은 CPU에 둡니다.')
else:
    print('CPU 환경이므로 데이터셋은 CPU tensor 캐시를 사용합니다.')

print('GPU 데이터 캐싱:', cache_data_on_device)


## 5. 샘플 이미지 확인


In [ ]:
def image_for_plot(flat_image):
    return flat_image.detach().cpu().view(3, 32, 32).permute(1, 2, 0).clamp(0, 1)

print('이미지 배치 형태 예시:', (batch_size, 3, 32, 32))
print('라벨 배치 형태 예시:', (batch_size,))
print('첫 번째 이미지 클래스 이름:', class_names[train_labels[0].detach().cpu().item()])

plt.figure(figsize=(3, 3))
plt.imshow(image_for_plot(train_images[0]))
plt.title(f'Label: {class_names[train_labels[0].detach().cpu().item()]}')
plt.axis('off')
plt.show()


## 6. 원본과 동일한 MLP 모델 정의

아래 계층 크기는 원본과 동일합니다. 모델 크기를 줄이지 않습니다.


In [ ]:
class CIFAR10MLP(nn.Module):
    def __init__(self):
        super(CIFAR10MLP, self).__init__()
        self.fc1 = nn.Linear(32 * 32 * 3, 2500)
        self.fc2 = nn.Linear(2500, 1500)
        self.fc3 = nn.Linear(1500, 500)
        self.fc4 = nn.Linear(500, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = CIFAR10MLP().to(device)
print(model)
print(f'학습 파라미터 수: {count_trainable_params(model):,}')


## 7. 컴파일과 학습 알고리즘 설정

모델 구조는 그대로 두고 optimizer와 스케줄링 알고리즘을 변경합니다. Adam은 파라미터마다 1차/2차 모멘트를 관리하므로 큰 MLP에서는 optimizer step 비용이 큽니다. 여기서는 더 가벼운 SGD + Momentum + Nesterov를 사용합니다.


In [ ]:
def autocast_context():
    if use_amp:
        return torch.cuda.amp.autocast()
    return nullcontext()


def maybe_compile_model(model, example_batch):
    # Windows나 CPU에서는 torch.compile이 오히려 느리거나 실패할 수 있어 안전하게 제한합니다.
    use_compile = hasattr(torch, 'compile') and device.type == 'cuda' and os.name != 'nt'
    if not use_compile:
        return model, False

    try:
        compiled_model = torch.compile(model, mode='reduce-overhead')
        model.eval()
        with torch.inference_mode():
            with autocast_context():
                _ = compiled_model(example_batch[:1].to(device))
        model.train()
        return compiled_model, True
    except Exception as exc:
        print('torch.compile 사용 실패, eager 모드로 진행합니다:', repr(exc))
        return model, False

example_batch = train_images[:batch_size]
train_model, compile_enabled = maybe_compile_model(model, example_batch)
print('torch.compile 사용:', compile_enabled)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=0.05,
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True,
    foreach=True,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


## 8. 빠른 미니배치 학습 루프

DataLoader 대신 이미 캐싱된 텐서를 직접 slicing/indexing합니다. `batch_size=50`은 그대로 유지합니다.


In [ ]:
def iter_minibatches(images, labels, batch_size, shuffle):
    n = labels.size(0)
    if shuffle:
        order = torch.randperm(n, device=labels.device)
    else:
        order = None

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        if order is None:
            batch_images = images[start:end]
            batch_labels = labels[start:end]
        else:
            batch_indices = order[start:end]
            batch_images = images.index_select(0, batch_indices)
            batch_labels = labels.index_select(0, batch_indices)
        yield batch_images, batch_labels


def move_batch_if_needed(images, labels):
    if images.device == device:
        return images, labels
    return images.to(device, non_blocking=True), labels.to(device, non_blocking=True)


def train_one_epoch(model, images, labels, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_images, batch_labels in iter_minibatches(images, labels, batch_size, shuffle=True):
        batch_images, batch_labels = move_batch_if_needed(batch_images, batch_labels)
        optimizer.zero_grad(set_to_none=True)

        with autocast_context():
            outputs = model(batch_images)
            loss = criterion(outputs, batch_labels)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        current_batch_size = batch_labels.size(0)
        running_loss += loss.detach().item() * current_batch_size
        correct += (outputs.argmax(dim=1) == batch_labels).sum().item()
        total += current_batch_size

    return running_loss / total, correct / total


def evaluate(model, images, labels, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for batch_images, batch_labels in iter_minibatches(images, labels, batch_size, shuffle=False):
            batch_images, batch_labels = move_batch_if_needed(batch_images, batch_labels)

            with autocast_context():
                outputs = model(batch_images)
                loss = criterion(outputs, batch_labels)

            current_batch_size = batch_labels.size(0)
            running_loss += loss.item() * current_batch_size
            correct += (outputs.argmax(dim=1) == batch_labels).sum().item()
            total += current_batch_size

    return running_loss / total, correct / total


## 9. 50 epoch 학습 실행

학습 epoch는 정확히 50번 수행합니다. 검증은 평가 오버헤드를 줄이기 위해 5 epoch마다 수행하고, 마지막 epoch에서는 반드시 수행합니다.


In [ ]:
train_losses = []
train_accuracies = []
val_epochs = []
val_losses = []
val_accuracies = []

best_val_acc = -1.0
best_state = copy.deepcopy(model.state_dict())
start_time = time.perf_counter()

for epoch in range(1, epochs + 1):
    epoch_start = time.perf_counter()

    train_loss, train_acc = train_one_epoch(
        train_model,
        train_images,
        train_labels,
        criterion,
        optimizer,
        device,
    )
    scheduler.step()

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    should_evaluate = (epoch % EVAL_EVERY == 0) or (epoch == epochs)
    if should_evaluate:
        val_loss, val_acc = evaluate(train_model, val_images, val_labels, criterion, device)
        val_epochs.append(epoch)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        val_message = f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | '
    else:
        val_message = 'Val: skip | '

    elapsed = time.perf_counter() - epoch_start
    print(
        f'Epoch [{epoch}/{epochs}] | '
        f'Train Loss: {train_loss:.4f} | '
        f'Train Acc: {train_acc:.4f} | '
        f'{val_message}'
        f'{elapsed:.1f}s'
    )

model.load_state_dict(best_state)
print(f'총 학습 시간: {time.perf_counter() - start_time:.1f}초')
print(f'최고 검증 정확도: {best_val_acc * 100:.2f}%')


## 10. 학습 결과 시각화


In [ ]:
epoch_axis = range(1, len(train_losses) + 1)

fig, loss_ax = plt.subplots(figsize=(10, 6))
acc_ax = loss_ax.twinx()

loss_ax.plot(epoch_axis, train_losses, label='train loss')
acc_ax.plot(epoch_axis, train_accuracies, label='train acc')

if val_losses:
    loss_ax.plot(val_epochs, val_losses, marker='o', label='val loss')
    acc_ax.plot(val_epochs, val_accuracies, marker='s', label='val acc')

loss_ax.set_xlabel('epoch')
loss_ax.set_ylabel('loss')
acc_ax.set_ylabel('accuracy')
loss_ax.legend(loc='upper left')
acc_ax.legend(loc='lower left')
plt.title('CIFAR-10 MLP Algorithm-Optimized Training History')
plt.show()


## 11. 테스트 데이터셋 최종 평가


In [ ]:
test_loss, test_acc = evaluate(train_model, test_images, test_labels, criterion, device)
print(f'test loss: {test_loss:.4f}')
print(f'accuracy: {test_acc * 100:.2f}%')


## 12. 예측 결과 확인


In [ ]:
train_model.eval()
preview_images = test_images[:10]
preview_labels = test_labels[:10]
preview_images_device, _ = move_batch_if_needed(preview_images, preview_labels)

with torch.inference_mode():
    with autocast_context():
        outputs = train_model(preview_images_device)
    predictions = outputs.argmax(dim=1).detach().cpu()

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(image_for_plot(preview_images[i]))
    pred_name = class_names[predictions[i].item()]
    true_name = class_names[preview_labels[i].detach().cpu().item()]
    plt.title(f'Pred: {pred_name}\nTrue: {true_name}')
    plt.axis('off')

plt.tight_layout()
plt.show()


## 정리

이 버전은 MLP 구조와 배치 크기를 건드리지 않았습니다. 원본과 같은 큰 MLP를 `batch_size=50`으로 50 epoch 학습해야 하므로 연산량 자체는 큽니다. 대신 반복 데이터 변환, DataLoader 오버헤드, 매 배치 전송 비용, Adam optimizer 비용을 줄이는 방식으로 같은 조건에서 더 빠르게 실행되도록 구성했습니다.
